# ⚖️ ISAC Coupling & Trade-off Governance Lab (Section II-E)

This notebook implements the **Quantitative 'Research Loop'** for **Section II-E (ISAC Coupling and Trade-off Foundations)**.
It combines **Regex Mining** (for coverage) with **Groq LLM Verification** (for semantic precision) to enforce strict governance on trade-off taxonomy (e.g., distinguishing *Resource Coupling* vs *Algorithmic Coupling*, *Pareto* vs *Weighted Sum*).

## Research Objectives (Section II-E)
1. **Detection**: Identify coupling mechanisms and trade-offs between Comm and Sensing.
2. **Governance Check**: Validate canonical taxonomy (Coupling Families, Trade-off Axes, Method Tags).
3. **Evidence Extraction**: Extract precise quotes and locators for the manuscript.

## Target Scope (II-E)
- **Resource Coupling**: Time/Freq/Power sharing, Bandwidth Split.
- **Waveform Coupling**: Joint Waveform Design, Dual-Function Waveforms.
- **Hardware Coupling**: Shared Transceiver, Shared OPA/RIS.
- **Algorithmic Coupling**: Joint Processing, Sensing-Aided Comm, Comm-Aided Sensing.
- **Multi-Objective Optimization**: Pareto Fronts, Weighted Sum, ε-Constraint.


In [1]:
# @title 1. Install & Setup
!pip install -q groq

from google.colab import drive
import os
import glob
import json
import csv
import re
from collections import Counter
from groq import Groq
from google.colab import userdata

# 1.1 Mount Drive & Set Base Dir
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST"

if os.path.exists(BASE_DIR):
    os.chdir(BASE_DIR)
    print(f"✅ Working Directory set to: {os.getcwd()}")
else:
    print(f"❌ Path not found: {BASE_DIR}. Please check your Drive structure.")

# 1.2 Load API Key
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    print("🔑 Groq API Key loaded.")
except Exception as e:
    print(f"⚠️ Error: {e}. Ensure 'GROQ_API_KEY' is in Colab Secrets.")

# 1.3 Define Output Paths
OUTPUT_DIR = "analysis/II_ev_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)
EVIDENCE_CSV = os.path.join(OUTPUT_DIR, "section2E_evidence.csv")
GOVERNANCE_DOC = "analysis/II_trade_gov_2E.md"
PATCH_NOTES = os.path.join(OUTPUT_DIR, "patch_notes_for_writing_2E.md")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Working Directory set to: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST
🔑 Groq API Key loaded.


In [2]:
# @title 2. Data Loader (Recursive)
def load_processed_markdowns(target_ids=None, limit=None):
    """
    Loads markdown files recursively from 'data/proc_markdowns'.
    Matches the logic of previous Governance Labs.
    """
    search_path = os.path.join("data", "processed_markdowns")

    # Recursive search for all .md files
    all_files = glob.glob(os.path.join(search_path, "**", "*.md"), recursive=True)

    # Filter for O_ISAC or COMST files
    valid_files = [f for f in all_files if "O_ISAC" in f or "COMST" in f]

    # ID Filtering
    selected_files = []
    if target_ids:
        print(f"Applying filter for {len(target_ids)} Target IDs...")
        for f_path in valid_files:
            p_id = os.path.basename(f_path).replace('.md', '')
            if p_id in target_ids:
                selected_files.append(f_path)
    else:
        selected_files = valid_files

    if limit:
        selected_files = selected_files[:limit]

    print(f"Found {len(valid_files)} total files. Loading {len(selected_files)} for analysis.")

    data = []
    for f_path in selected_files:
        p_id = os.path.basename(f_path).replace('.md', '')
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                content = f.read()
                data.append((p_id, content))
        except Exception as e:
            print(f"Error reading {f_path}: {e}")

    return data

In [3]:
# @title 3. Define II-E Research Agent (Coupling & Trade-off Extraction)
def analyze_tradeoff_governance(paper_text, paper_id):
    """
    Asks LLM to extract ISAC Coupling and Trade-off evidence from the paper.
    Includes 'Reasoning' for Evidence Locking.
    """

    system_prompt = """
    You are a Senior ISAC Architect auditing O-ISAC papers for Section II-E (Coupling & Trade-offs).
    Your goal is to extract EXACT evidence of:
    1. **Coupling Mechanisms**: How Comm and Sensing share resources/hardware/waveforms.
    2. **Trade-off Formulations**: Explicit axes (e.g., Rate vs CRB, BER vs Pd).
    3. **Optimization Methods**: Pareto, Weighted Sum, Alternating Optimization, etc.

    # COUPLING FAMILIES (pick one):
    - resource_coupling: time/freq/power/bandwidth sharing
    - waveform_coupling: joint/dual-function waveform
    - hardware_coupling: shared transceiver/detector/OPA
    - algorithmic_coupling: joint processing, sensing-aided comm
    - geometric_coupling: turbulence/pointing/multipath affects both
    - multi_objective: Pareto, weighted sum, epsilon-constraint

    # CRITICAL RULES:
    - Distinguish 'trade-off statement' (explicit) vs 'potential trade-off' (hypothetical).
    - Identify BOTH axes (comm metric vs sensing metric) if present.
    - IGNORE Related Work mentions.

    # OUTPUT FORMAT:
    Return a JSON object with a 'reasoning' string and a 'findings' list.
    Example:
    {
      "reasoning": "The authors formulate a joint optimization in Section IV, trading off spectral efficiency against ranging accuracy using weighted sum.",
      "findings": [
         { "coupling_family": "multi_objective", "axis_x": "spectral_efficiency", "axis_y": "ranging_accuracy", "method": "weighted_sum", "evidence": "we optimize a weighted sum of SE and ranging RMSE..." }
      ]
    }
    """

    user_prompt = f"""
    Paper ID: {paper_id}

    Analyze this text strictly. DO NOT hallucinate. If no explicit trade-off is formulated, say 'NONE detected'.

    Text Content (First 35k chars):
    {paper_text[:35000]}
    """

    try:
        completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            model="llama-3.3-70b-versatile",
            response_format={"type": "json_object"},
            temperature=0
        )
        return json.loads(completion.choices[0].message.content)
    except Exception as e:
        return {"error": str(e), "reasoning": "Fail", "findings": []}


In [4]:
# @title 4. Execute Research Loop

# ==========================================
# CONFIGURATION
# ==========================================
# Set to None for full corpus scan, or specify IDs for targeted analysis
TARGET_PAPERS = None  # e.g., ['O_ISAC_029', 'O_ISAC_051', 'O_ISAC_055']
LIMIT = None          # e.g., 10 for quick test
OUTPUT_CSV = "analysis/II_ev_v2/section2E_evidence_LLM.csv"
# ==========================================

# 1. Load Papers
papers = load_processed_markdowns(target_ids=TARGET_PAPERS, limit=LIMIT)

# 2. Run Agent
all_findings = []
print(f"\n🚀 Starting II-E Trade-off Analysis on {len(papers)} papers...\n")

for pid, text in papers:
    print(f"Processing {pid}...")
    result = analyze_tradeoff_governance(text, pid)

    reasoning = result.get("reasoning", "No reasoning provided.")
    findings = result.get("findings", [])

    print(f"  🧠 Agent Reasoning: {reasoning[:150]}...")

    for f in findings:
        f["paper_id"] = pid
        f["full_reasoning"] = reasoning
        all_findings.append(f)
        print(f"     -> ⚖️ Found {f.get('coupling_family')}: {f.get('axis_x')} vs {f.get('axis_y')}")
    print("-"*40)

# 3. Export Results
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
if all_findings:
    keys = ["paper_id", "coupling_family", "axis_x", "axis_y", "method", "evidence", "full_reasoning"]
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_findings)
    print(f"\n✅ Saved {len(all_findings)} confirmed evidence rows to {OUTPUT_CSV}")
    # Show first few results
    import pandas as pd
    display(pd.read_csv(OUTPUT_CSV).head(10))
else:
    print("\n⚠️ No findings extracted.")

Found 313 total files. Loading 313 for analysis.

🚀 Starting II-E Trade-off Analysis on 313 papers...

Processing O_ISAC_029...
  🧠 Agent Reasoning: The authors propose a photonic-based THz ISAC system with full-photonic direct LFM reception and de-chirping for fiber-wireless networks. The system a...
     -> ⚖️ Found waveform_coupling: NONE detected vs NONE detected
----------------------------------------
Processing O_ISAC_029...
  🧠 Agent Reasoning: The authors propose a photonic-based THz ISAC system with full-photonic direct LFM reception and de-chirping for fiber-wireless networks. The system a...
     -> ⚖️ Found waveform_coupling: NONE detected vs NONE detected
----------------------------------------
Processing O_ISAC_001...
  🧠 Agent Reasoning: The authors discuss the trade-off between modulation index and system performance in terms of EVM, but do not explicitly formulate a trade-off between...
     -> ⚖️ Found waveform_coupling: modulation_index vs EVM
---------------------

,paper_id,coupling_family,axis_x,axis_y,method,evidence,full_reasoning
0,O_ISAC_029,waveform_coupling,NONE detected,NONE detected,NONE detected,No explicit trade-off formulation is presented...,The authors propose a photonic-based THz ISAC ...
1,O_ISAC_029,waveform_coupling,NONE detected,NONE detected,NONE detected,No explicit trade-off formulation is presented...,The authors propose a photonic-based THz ISAC ...
2,O_ISAC_001,waveform_coupling,modulation_index,EVM,NONE,The results reveal a trade-off between modulat...,The authors discuss the trade-off between modu...
3,O_ISAC_001,waveform_coupling,modulation index,EVM,NONE,The results reveal a trade-off between modulat...,The authors discuss the trade-off between modu...
4,O_ISAC_002,waveform_coupling,NONE detected,NONE detected,NONE detected,The photonic THz-ISAC system needs to effectiv...,The authors discuss the integration of sensing...
5,O_ISAC_002,resource_coupling,NONE detected,NONE detected,NONE detected,Waveform design must integrate these considera...,The authors discuss the integration of sensing...
6,O_ISAC_002,waveform_coupling,NONE detected,NONE detected,NONE detected,The paper focuses on integrated waveform desig...,The authors discuss the integration of sensing...
7,O_ISAC_003,NONE detected,NONE detected,NONE detected,NONE detected,No explicit trade-off formulation is found in ...,The authors analyze the visible light-based in...
8,O_ISAC_003,NONE detected,NONE detected,NONE detected,NONE detected,No explicit trade-off formulation is found in ...,The authors analyze the channel characteristic...
9,O_ISAC_004,hardware_coupling,NONE detected,NONE detected,NONE detected,The system integrates optical fiber sensing an...,The authors demonstrate an adiabatic-tapered f...


In [5]:
# @title 5. Generate Governance Artifacts (II-E)
def generate_governance_doc(findings):
    if not findings: return

    families = [f.get('coupling_family', 'unknown') for f in findings]
    methods = [f.get('method', 'unknown') for f in findings]
    axes_x = [f.get('axis_x', 'unknown') for f in findings]
    axes_y = [f.get('axis_y', 'unknown') for f in findings]

    family_counts = Counter(families)
    method_counts = Counter(methods)
    axis_pairs = Counter(zip(axes_x, axes_y))

    content = "# Section II-E: Trade-off Governance (Draft)\n\n"

    content += "## 1. Coupling Families (Detected Usage)\n"
    for fam, cnt in family_counts.most_common():
        content += f"- **{fam}**: {cnt} occurrences\n"

    content += "\n## 2. Optimization Methods (Detected Usage)\n"
    for m, cnt in method_counts.most_common():
        content += f"- **{m}**: {cnt} occurrences\n"

    content += "\n## 3. Trade-off Axis Pairs (Top 15)\n"
    for (ax, ay), cnt in axis_pairs.most_common(15):
        content += f"- **{ax}** vs **{ay}**: {cnt}\n"

    content += "\n## 4. Do-Not-Conflate Rules\n"
    content += "- **Resolution** (Δr) ≠ **Accuracy** (σ_r / RMSE)\n"
    content += "- **CRB** (theoretical bound) ≠ **Empirical RMSE**\n"
    content += "- **Pd/Pfa** (detection) ≠ **σ_r** (estimation)\n"
    content += "- **OSNR** (optical plane) ≠ **SNR** (electrical plane)\n"
    content += "- **Pareto** (multi-objective) ≠ **Weighted Sum** (scalarization)\n"

    with open(GOVERNANCE_DOC, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Generated {GOVERNANCE_DOC}")

def generate_patch_notes(findings):
    if not findings: return

    content = "# Patch Notes for Section II-E Writing\n\n"

    # Select up to 25 strong evidence items
    selected = findings[:25]

    for i, f in enumerate(selected, 1):
        pid = f.get('paper_id', '?')
        fam = f.get('coupling_family', '?')
        ax = f.get('axis_x', '?')
        ay = f.get('axis_y', '?')
        ev = f.get('evidence', '')[:100]
        content += f"{i}. **[{pid}]** {fam}: {ax} vs {ay}. \"{ev}...\"\n\n"

    with open(PATCH_NOTES, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Generated {PATCH_NOTES}")

if all_findings:
    generate_governance_doc(all_findings)
    generate_patch_notes(all_findings)

Generated analysis/II_trade_gov_2E.md
Generated analysis/II_ev_v2/patch_notes_for_writing_2E.md


In [6]:
# @title 6. Quality Control Dashboard
if all_findings:
    import pandas as pd
    df = pd.DataFrame(all_findings)

    print("="*50)
    print("QC DASHBOARD - Section II-E")
    print("="*50)
    print(f"Total Evidence Rows: {len(df)}")
    print(f"Unique Papers: {df['paper_id'].nunique()}")
    print("\n--- Coupling Family Distribution ---")
    print(df['coupling_family'].value_counts())
    print("\n--- Method Distribution ---")
    if 'method' in df.columns:
        print(df['method'].value_counts())
    print("="*50)
else:
    print("No findings to analyze.")

QC DASHBOARD - Section II-E
Total Evidence Rows: 370
Unique Papers: 200

--- Coupling Family Distribution ---
coupling_family
waveform_coupling       150
resource_coupling        63
NONE detected            50
algorithmic_coupling     38
multi_objective          37
hardware_coupling        32
Name: count, dtype: int64

--- Method Distribution ---
method
NONE detected                         198
NONE                                   36
weighted_sum                           33
trade-off                               5
elliptical search algorithm             4
                                     ... 
integrated_optoelectronic_chip          1
adjusting_transmitter_bias_voltage      1
multi_frequency_fusion                  1
adaptive_waveform_adjustment            1
frequency-division_multiplexing         1
Name: count, Length: 66, dtype: int64
